<a href="https://colab.research.google.com/github/piriyadharshiniN/24ADI006-DL-Lab/blob/main/DL_Exp_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Part A - Environment setup
%pip install -q -U "transformers>=4.45.0" accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.8 MB/s eta 0:00:00


In [2]:
# Import libraries and check hardware
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU; generation may be slower.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [9]:
# Load the pretrained model and tokenizer
print("Name: Piriyadharshini N\nRoll no.:24BAD086\nExp: 10\n")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto"
)
model.eval()

print("Model loaded:", MODEL_ID)
print("Model device:", model.device)

Name: Piriyadharshini N
Roll no.:24BAD086
Exp: 10



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Model device: cuda:0


In [4]:
# Part B - Prompt handling and response generation
SYSTEM_PROMPT = "You are a helpful, polite assistant. Answer clearly and concisely. If unsure, say so."

def generate_response(message, history=None, max_new_tokens=128):
    if message is None or not str(message).strip():
        return "Please enter a message."

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for item in (history or [])[-8:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role in ("user", "assistant") and isinstance(content, str) and content.strip():
                messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": str(message).strip()})

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return response or "I could not generate a response. Please try another prompt."
    except Exception as exc:
        return f"Error while generating the response: {type(exc).__name__}: {exc}"

print("Response function defined.")

Response function defined.


In [5]:
# Test one simple prompt
prompt = "What is artificial intelligence?"
print("Prompt:", prompt)
print("Response:", generate_response(prompt))

Prompt: What is artificial intelligence?
Response: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It involves developing algorithms, systems, or software that can perform tasks that typically require human intelligence, such as recognizing patterns, learning from experience, and making decisions based on information. AI uses algorithms, statistical models, and machine learning techniques to process data, identify trends, and make predictions or decisions. This technology has been used across various industries for applications ranging from self-driving cars to healthcare diagnostics, financial forecasting, and more.


In [6]:
# Test five different prompts
prompts = [
    "Explain machine learning in simple words.",
    "Give three tips for preparing for an exam.",
    "Write a short Python program to add two numbers.",
    "What is the difference between AI and machine learning?",
    "Give a two-sentence summary of renewable energy."
]

for i, prompt in enumerate(prompts, start=1):
    print(f"\nTEST {i}: {prompt}")
    print(generate_response(prompt, history=[], max_new_tokens=80))
    print("-" * 55)


TEST 1: Explain machine learning in simple words.
Machine learning is a subset of artificial intelligence that involves algorithms that enable computers to learn from data without being explicitly programmed. It's like teaching a computer new tricks based on how it learns from its experiences with similar tasks. Machine learning can be used for a wide range of applications including image recognition, natural language processing, predictive analytics, and more. Essentially, it allows computers to automatically improve their performance by analyzing
-------------------------------------------------------

TEST 2: Give three tips for preparing for an exam.
1. **Prioritize Review**: Focus on reviewing the key concepts and formulas that will be tested in your exam. This can help you better understand the material and retain information more effectively.

2. **Create a Study Schedule**: Plan out how much time you need to study each day or week, taking into account different subjects and th

In [7]:
# Part C - Building chatbot Interface

def chatbot_fn(message, history):
    return generate_response(message, history=history)

demo = gr.ChatInterface(
    fn=chatbot_fn,
    title="Simple LLM Chatbot",
    description="A chatbot powered by a pretrained Transformer model.",
    examples=[
        "Explain neural networks in simple words.",
        "Give me a study timetable for one day.",
        "Write a Python function to check an even number.",
        "What is climate change?"
    ],
    save_history=True
)

print("Gradio interface created.")

Gradio interface created.


In [8]:
# Part D - Testing and Deployment of the chatbot
# share=True provides a temporary link for demonstration.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://29eae18ebab0b30ba7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
